# Boca Juniors 2026 | Comparación de ciclos

El análisis abarca los 32 partidos de zonas que sumaron para la tabla anual.
Los balances se calculan desde los marcadores. La asignación de los
partidos a cada ciclo está documentada en [fuentes](../docs/fuentes.md).

In [10]:
from pathlib import Path

import pandas as pd

from lpf_analytics.resultados import resumen_equipo

actual = Path.cwd()
raiz = next(
    carpeta
    for carpeta in (actual, *actual.parents)
    if (carpeta / "pyproject.toml").is_file()
)
equipo = "Boca Juniors"
partidos = pd.read_csv(
    raiz / "data/raw/LPF_2026_partidos.csv", encoding="utf-8-sig"
)
es_boca = partidos[["local", "visitante"]].eq(equipo).any(axis=1)
boca = partidos.loc[es_boca].copy()
boca["condicion"] = boca["local"].eq(equipo).map({
    True: "Local",
    False: "Visitante",
})


def balance_por(claves):
    filas = []
    for valores, grupo in boca.groupby(claves, dropna=False):
        balance = resumen_equipo(grupo, equipo)
        filas.append({**dict(zip(claves, valores)), **balance})

    tabla = pd.DataFrame(filas)
    tabla["DG"] = tabla["GF"] - tabla["GC"]
    tabla["PTS/PJ"] = (tabla["PTS"] / tabla["PJ"]).round(2)
    return tabla

## Balance por ciclo

In [11]:
balance_por(["torneo", "dt_boca"])

,torneo,dt_boca,PJ,PG,PE,PP,PTS,GF,GC,DG,PTS/PJ
0,Apertura,Claudio Úbeda,16,8,6,2,30,22,9,13,1.88
1,Clausura,Rodolfo Arruabarrena,16,9,6,1,33,24,14,10,2.06


## Balance por localía

In [12]:
balance_por(["torneo", "dt_boca", "condicion"])

,torneo,dt_boca,condicion,PJ,PG,PE,PP,PTS,GF,GC,DG,PTS/PJ
0,Apertura,Claudio Úbeda,Local,8,3,5,0,14,8,3,5,1.75
1,Apertura,Claudio Úbeda,Visitante,8,5,1,2,16,14,6,8,2.00
2,Clausura,Rodolfo Arruabarrena,Local,8,7,1,0,22,13,3,10,2.75
3,Clausura,Rodolfo Arruabarrena,Visitante,8,2,5,1,11,11,11,0,1.38


## Qué cambió entre los ciclos

Boca sumó 30 puntos con Úbeda y 33 con Arruabarrena en 16 partidos por ciclo. De local pasó de 14 a 22 puntos y de tres a siete victorias en ocho partidos. De visitante pasó de 16 a 11 puntos.

En el Clausura marcó dos goles más y recibió cinco más. Todo el aumento de goles recibidos se dio de visitante, donde pasó de 6 a 11. De local recibió 3 en cada ciclo.

La comparación es descriptiva. Cada ciclo coincide con un torneo distinto, por lo que el cambio de entrenador y el cambio de torneo se observan juntos.